In [ ]:
### Geolibraries
import geopandas as gpd
import osmnx as ox
import contextily as ctx; import basemaps


# General tools
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import pyarrow.parquet as pq

from h3 import h3
from shapely.geometry import Polygon
import mapclassify as mc


In [ ]:
final_df_bike = pd.read_parquet("./output/bike_expenditure_weekly_nearest.parquet")

In [ ]:
final_df_bike.columns

In [ ]:
final_df_car = pd.read_parquet("./output/car_expenditure_weekly_nearest.parquet")

In [ ]:
final_df_car.columns

In [ ]:
final_df_pt = pd.read_parquet("./output/PT_expenditure_weekly_nearest.parquet")

In [ ]:
final_df_pt.columns

In [ ]:
hex_pt = (
    final_df_pt
    .dropna(subset=["weekly_co2_expenditure"])
    .groupby("home_gid9")
    .agg(
        avg_co2=("weekly_co2_expenditure", "mean"),
        n_users=("user_id", "count")
    )
    .reset_index()
)
hex_pt = hex_pt[hex_pt["n_users"] >= 3]

In [ ]:
import h3
import geopandas as gpd
from shapely.geometry import Polygon

def h3_to_polygon(h):
    boundary = h3.h3_to_geo_boundary(h, geo_json=True)
    return Polygon(boundary)

hex_pt["geometry"] = hex_pt["home_gid9"].apply(h3_to_polygon)

gdf_pt = gpd.GeoDataFrame(hex_pt, geometry="geometry", crs="EPSG:4326")


In [ ]:
gdf_pt = gdf_pt.to_crs(3857)
scheme = mc.NaturalBreaks(gdf_pt["avg_co2"], k=6)  # 5 classes is ideal
gdf_pt["avg_co2_kg"] = gdf_pt["avg_co2"] / 1000

In [ ]:
gdf_pt.sort_values("avg_co2_kg").head(1900)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

gdf_pt.plot(
    column="avg_co2_kg",
    cmap="RdBu_r",
    scheme="NaturalBreaks",
    classification_kwds={"k": 6},
    linewidth=0.2,
    edgecolor="black",
    alpha=0.9,
    legend=True,
    legend_kwds={
        "title": "Weekly CO₂ (kg)",
        "loc": "upper left"
    },
    ax=ax
)

ctx.add_basemap(ax, source=basemaps.POSITRON)

ax.set_axis_off()
ax.set_title("Average Weekly CO₂ per Resident Hexagon", fontsize=14)

plt.tight_layout()
plt.show()



In [ ]:
# 1. aggregate mean CO₂ and user count
car_hex_agg = (
    final_df_car
    .groupby("home_gid9", as_index=False)
    .agg(
        avg_co2=("weekly_co2_expenditure_car", "mean"),
        n_users=("user_id", "count")
    )
)

# 2. filter to hexes with at least 3 users
car_hex_agg = car_hex_agg[car_hex_agg["n_users"] >= 3]

In [ ]:
def h3_to_polygon(h):
    boundary = h3.h3_to_geo_boundary(h, geo_json=True)
    return Polygon(boundary)

gdf_car = gpd.GeoDataFrame(
    car_hex_agg,
    geometry=car_hex_agg["home_gid9"].apply(h3_to_polygon),
    crs="EPSG:4326"  # lat/lon
)

In [ ]:
gdf_car["avg_co2_kg"] = gdf_car["avg_co2"] / 1000
gdf_car = gdf_car.to_crs(3857)

In [ ]:
import contextily as ctx; import basemaps
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 8))

gdf_car.plot(
    column="avg_co2_kg",
    cmap="RdBu_r",
    scheme="NaturalBreaks",
    classification_kwds={"k": 6},
    linewidth=0.2,
    edgecolor="black",
    alpha=0.9,
    legend=True,
    legend_kwds={
        "title": "Weekly CO₂ (kg)",
        "loc": "upper left"
    },
    ax=ax
)

ctx.add_basemap(ax, source=basemaps.POSITRON)

ax.set_axis_off()
ax.set_title(
    "Average Weekly CO₂ per Resident Hexagon (Car)",
    fontsize=14
)

plt.tight_layout()
plt.show()


In [ ]:
# Aggregate mean CO₂ per hex and count users
bike_hex_agg = (
    final_df_bike
    .groupby("home_gid9", as_index=False)
    .agg(
        avg_co2=("weekly_co2_expenditure_bike", "mean"),
        n_users=("user_id", "count")
    )
)

# Filter hexes with at least 3 users
bike_hex_agg = bike_hex_agg[bike_hex_agg["n_users"] >= 3]


In [ ]:


def h3_to_polygon(h):
    boundary = h3.h3_to_geo_boundary(h, geo_json=True)
    return Polygon(boundary)

gdf_bike = gpd.GeoDataFrame(
    bike_hex_agg,
    geometry=bike_hex_agg["home_gid9"].apply(h3_to_polygon),
    crs="EPSG:4326"  # lat/lon
)


In [ ]:
gdf_bike["avg_co2_kg"] = gdf_bike["avg_co2"] / 1000
gdf_bike = gdf_bike.to_crs(3857)


In [ ]:
import contextily as ctx; import basemaps
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 8))

gdf_bike.plot(
    column="avg_co2_kg",
    cmap="RdBu_r",
    scheme="NaturalBreaks",
    classification_kwds={"k": 6},
    linewidth=0.2,
    edgecolor="black",
    alpha=0.9,
    legend=True,
    legend_kwds={
        "title": "Weekly CO₂ (kg)",
        "loc": "upper left"
    },
    ax=ax
)

ctx.add_basemap(ax, source=basemaps.POSITRON)

ax.set_axis_off()
ax.set_title(
    "Average Weekly CO₂ per Resident Hexagon",
    fontsize=14
)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import contextily as ctx; import basemaps
import pandas as pd
import numpy as np

# --- Bins and labels ---
bins = [0, 1, 3, 7, 10, 15, np.inf]
labels = ["0-1", "1-3", "3-7", "7-10", "10-15", "15+"]
colors_dict = {
    "0-1": "#4575b4",
    "1-3": "#91bfdb",
    "3-7": "#fee090",
    "7-10": "#fc8d59",
    "10-15": "#d73027",
    "15+": "#a50026"
}

# --- Categorize CO₂ ---
def categorize_co2(gdf, col="avg_co2_kg"):
    gdf = gdf.copy()
    gdf["co2_cat"] = pd.cut(gdf[col], bins=bins, labels=labels, include_lowest=True)
    return gdf

gdf_pt = categorize_co2(gdf_pt)
gdf_car = categorize_co2(gdf_car)
gdf_bike = categorize_co2(gdf_bike)

# --- Plot vertically with categorical legend ---
fig, axes = plt.subplots(3, 1, figsize=(12, 24))  # taller figure

for ax, gdf, title in zip(
    axes,
    [gdf_pt, gdf_car, gdf_bike],
    ["PT", "Car", "Bike"]
):
    for cat, color in colors_dict.items():
        subset = gdf[gdf["co2_cat"] == cat]
        if not subset.empty:
            subset.plot(
                color=color,
                linewidth=0.2,
                edgecolor="black",
                alpha=0.9,
                ax=ax
            )

    ctx.add_basemap(ax, source=basemaps.POSITRON)
    ax.set_axis_off()
    ax.set_title(f"Average Weekly CO₂ per Resident Hexagon ({title})", fontsize=14)

    # --- Add categorical legend manually ---
    handles = [plt.Line2D([0], [0], marker='s', color=color, linestyle='', markersize=10)
               for color in colors_dict.values()]
    ax.legend(handles, labels, title="Weekly CO₂ (kg)", loc="upper right", frameon=True)

plt.tight_layout()
plt.show()



In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import contextily as ctx; import basemaps
import pandas as pd
import numpy as np

# -----------------------------
# Bins and labels (4 categories)
# -----------------------------
bins = [0, 1, 3, 7, np.inf]
labels = ["0–1", "1–3", "3–7", "7+"]

# Darker = lower emissions
colors_dict = {
    "0–1": "#00441b",   # dark green (lowest emissions)
    "1–3": "#41ab5d",   # medium green
    "3–7": "#c7e9c0",   # light green
    "7+": "#d73027"     # red (above budget)
}

# -----------------------------
# Categorize CO₂ values
# -----------------------------
def categorize_co2(gdf, col="avg_co2_kg"):
    gdf = gdf.copy()
    gdf["co2_cat"] = pd.cut(
        gdf[col],
        bins=bins,
        labels=labels,
        include_lowest=True
    )
    return gdf

gdf_pt = categorize_co2(gdf_pt)
gdf_car = categorize_co2(gdf_car)
gdf_bike = categorize_co2(gdf_bike)

# -----------------------------
# Create figure
# -----------------------------
fig, axes = plt.subplots(3, 1, figsize=(12, 24))

for ax, gdf, title in zip(
    axes,
    [gdf_pt, gdf_car, gdf_bike],
    ["Public Transport", "Car", "Bike"]
):
    
    for cat in labels:
        subset = gdf[gdf["co2_cat"] == cat]
        if not subset.empty:
            subset.plot(
                color=colors_dict[cat],
                linewidth=0.2,
                edgecolor="black",
                alpha=0.9,
                ax=ax
            )

    ctx.add_basemap(ax, source=basemaps.POSITRON)

    ax.set_axis_off()
    ax.set_title(
        f"Average Weekly CO₂ per Resident Hexagon ({title})",
        fontsize=14
    )

    # Legend
    handles = [
        plt.Line2D([0], [0], marker='s',
                   color=colors_dict[label],
                   linestyle='',
                   markersize=10)
        for label in labels
    ]

    ax.legend(
        handles,
        labels,
        title="Weekly CO₂ (kg)",
        loc="upper right",
        frameon=True
    )

plt.tight_layout()

# -----------------------------
# SAVE AS ONE IMAGE
# -----------------------------
plt.savefig(
    "weekly_co2_maps.png",
    dpi=300,              # high resolution
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

In [ ]:
gdf_pt_nearest = gdf_pt.copy()
gdf_car_nearest = gdf_car.copy()
gdf_bike_nearest = gdf_bike.copy()

In [ ]:
gdf_car_nearest

## Typical

In [ ]:
final_df_car_typ = pd.read_parquet("./output/car_expenditure_weekly_typical.parquet")
final_df_pt_typ  = pd.read_parquet("./output/PT_expenditure_weekly_typical.parquet")
final_df_bike_typ = pd.read_parquet("./output/bike_expenditure_weekly_typical.parquet")


In [ ]:
final_df_bike_typ.columns

In [ ]:
final_df_pt_typ.columns

In [ ]:
final_df_car_typ.columns

In [ ]:
user_home_hex = (
    final_df_pt[["user_id", "home_gid9"]]
    .drop_duplicates()
)

In [ ]:
car_typ_hex = final_df_car_typ.merge(
    user_home_hex,
    on="user_id",
    how="left"
)

pt_typ_hex = final_df_pt_typ.merge(
    user_home_hex,
    on="user_id",
    how="left"
)

bike_typ_hex = final_df_bike_typ.merge(
    user_home_hex,
    on="user_id",
    how="left"
)

In [ ]:
pt_hex = (
    pt_typ_hex
    .groupby(["home_gid9", "Nimi", "Posnro"])["total_weekly_co2"]
    .agg(["mean", "count"])
    .reset_index()
)

pt_hex = pt_hex[pt_hex["count"] >= 3]
pt_hex["avg_co2_kg"] = pt_hex["mean"] / 1000

In [ ]:
car_hex = (
    car_typ_hex
    .groupby(["home_gid9", "Nimi", "Posnro"])["total_weekly_co2"]
    .agg(["mean", "count"])
    .reset_index()
)

car_hex = car_hex[car_hex["count"] >= 3]
car_hex["avg_co2_kg"] = car_hex["mean"] / 1000

In [ ]:
bike_hex = (
    bike_typ_hex
    .groupby(["home_gid9", "Nimi", "Posnro"])["total_weekly_co2"]
    .agg(["mean", "count"])
    .reset_index()
)

bike_hex = bike_hex[bike_hex["count"] >= 3]
bike_hex["avg_co2_kg"] = bike_hex["mean"] / 1000

In [ ]:
pt_hex.sort_values("avg_co2_kg").head(1200)

In [ ]:
def h3_to_polygon(h):
    # Returns list of (lat, lng) tuples
    boundary = h3.h3_to_geo_boundary(h, geo_json=True)  
    return Polygon(boundary)

In [ ]:
car_hex["geometry"] = car_hex["home_gid9"].apply(h3_to_polygon)
bike_hex["geometry"] = bike_hex["home_gid9"].apply(h3_to_polygon)
pt_hex["geometry"] = pt_hex["home_gid9"].apply(h3_to_polygon)

In [ ]:
car_gdf  = gpd.GeoDataFrame(car_hex,  geometry="geometry", crs="EPSG:4326").to_crs(3857)
bike_gdf = gpd.GeoDataFrame(bike_hex, geometry="geometry", crs="EPSG:4326").to_crs(3857)
pt_gdf   = gpd.GeoDataFrame(pt_hex,   geometry="geometry", crs="EPSG:4326").to_crs(3857)


In [ ]:
bins = [0, 1, 3, 7, 10, 15, 100]
labels = ["0–1", "1–3", "3–7", "7–10", "10–15", "15+"]

for gdf in [car_gdf, bike_gdf, pt_gdf]:
    gdf["co2_cat"] = pd.cut(
        gdf["avg_co2_kg"],
        bins=bins,
        labels=labels,
        include_lowest=True
    )


In [ ]:
colors = {
    "0–1": "#f7fbff",
    "1–3": "#deebf7",
    "3–7": "#c6dbef",
    "7–10": "#9ecae1",
    "10–15": "#6baed6",
    "15+": "#2171b5"
}


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
import pandas as pd

# -----------------------------
# Carbon budget thresholds
# -----------------------------
bins = [0, 1, 3, 7, np.inf]
labels = ["0–1", "1–3", "3–7", "7+"]

# darker green = lower emissions
colors_dict = {
    "0–1": "#00441b",   # dark green (lowest)
    "1–3": "#41ab5d",   # medium green
    "3–7": "#c7e9c0",   # light green
    "7+": "#d73027"     # red (above budget)
}

# -----------------------------
# Categorize CO₂
# -----------------------------
def categorize_co2(gdf, col="avg_co2_kg"):
    gdf = gdf.copy()
    gdf["co2_cat"] = pd.cut(
        gdf[col],
        bins=bins,
        labels=labels,
        include_lowest=True
    )
    return gdf

gdf_pt   = categorize_co2(pt_gdf)
gdf_car  = categorize_co2(car_gdf)
gdf_bike = categorize_co2(bike_gdf)

# -----------------------------
# Plot maps (vertical layout)
# -----------------------------
fig, axes = plt.subplots(3, 1, figsize=(12, 24))

for ax, gdf, title in zip(
    axes,
    [gdf_pt, gdf_car, gdf_bike],
    ["Public Transport", "Car", "Bike"]
):

    for cat in labels:
        subset = gdf[gdf["co2_cat"] == cat]
        if not subset.empty:
            subset.plot(
                color=colors_dict[cat],
                linewidth=0.2,
                edgecolor="black",
                alpha=0.9,
                ax=ax
            )

    ctx.add_basemap(ax, source=basemaps.POSITRON)

    ax.set_axis_off()
    ax.set_title(
        f"Weekly CO₂ per Resident Hexagon ({title})",
        fontsize=14
    )

    # Legend
    handles = [
        plt.Line2D([0], [0], marker='s',
                   color=colors_dict[label],
                   linestyle='',
                   markersize=10)
        for label in labels
    ]

    ax.legend(
        handles,
        labels,
        title="Weekly CO₂ (kg)",
        loc="upper right",
        frameon=True
    )

plt.tight_layout()

# -----------------------------
# Save as single high-res image
# -----------------------------
plt.savefig(
    "typical_case_weekly_co2_maps.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
import pandas as pd

# -----------------------------
# Carbon budget thresholds
# -----------------------------
bins = [0, 1, 3, 7, np.inf]
labels = ["0–1", "1–3", "3–7", "7+"]

colors_dict = {
    "0–1": "#00441b",
    "1–3": "#41ab5d",
    "3–7": "#c7e9c0",
    "7+": "#d73027"
}

# -----------------------------
# Categorize CO₂
# -----------------------------
def categorize_co2(gdf, col="avg_co2_kg"):
    gdf = gdf.copy()
    gdf["co2_cat"] = pd.cut(
        gdf[col],
        bins=bins,
        labels=labels,
        include_lowest=True
    )
    return gdf

gdf_pt   = categorize_co2(pt_gdf)
gdf_car  = categorize_co2(car_gdf)
gdf_bike = categorize_co2(bike_gdf)

# -----------------------------
# HORIZONTAL PLOT (poster style)
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# NEW ORDER
titles = ["Bike", "Public Transport", "Car"]
gdfs   = [gdf_bike, gdf_pt, gdf_car]

for ax, gdf, title in zip(axes, gdfs, titles):

    for cat in labels:
        subset = gdf[gdf["co2_cat"] == cat]
        if not subset.empty:
            subset.plot(
                color=colors_dict[cat],
                linewidth=0.1,
                edgecolor="none",
                alpha=0.9,
                ax=ax
            )

    ctx.add_basemap(ax, source=basemaps.POSITRON)

    ax.set_axis_off()
    ax.set_title(title, fontsize=16)

# -----------------------------
# SINGLE LEGEND (CLOSER)
# -----------------------------
handles = [
    plt.Line2D([0], [0], marker='s',
               color=colors_dict[label],
               linestyle='',
               markersize=10)
    for label in labels
]

fig.legend(
    handles,
    labels,
    title="Weekly CO₂ (kg)",
    loc="lower center",
    bbox_to_anchor=(0.5, 0.06),  # closer to maps
    ncol=4,
    frameon=False,
    fontsize=12,
    title_fontsize=12
)

# -----------------------------
# LAYOUT
# -----------------------------
plt.tight_layout(rect=[0, 0.06, 1, 1])  # less bottom space

# -----------------------------
# SAVE
# -----------------------------
plt.savefig(
    "co2_maps_horizontal.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

In [ ]:
gdf_bike

In [ ]:
gdf_bike_nearest

In [ ]:
bike_gap = gdf_bike[['home_gid9','avg_co2_kg','geometry']].merge(
    gdf_bike_nearest[['home_gid9','avg_co2_kg']],
    on='home_gid9',
    suffixes=('_typical','_nearest')
)

bike_gap['gap_bike'] = (
    bike_gap['avg_co2_kg_typical']
    - bike_gap['avg_co2_kg_nearest']
)

In [ ]:
pt_gap = gdf_pt[['home_gid9','avg_co2_kg','geometry']].merge(
    gdf_pt_nearest[['home_gid9','avg_co2_kg']],
    on='home_gid9',
    suffixes=('_typical','_nearest')
)

pt_gap['gap_pt'] = (
    pt_gap['avg_co2_kg_typical']
    - pt_gap['avg_co2_kg_nearest']
)


In [ ]:
car_gap = gdf_car[['home_gid9','avg_co2_kg','geometry']].merge(
    gdf_car_nearest[['home_gid9','avg_co2_kg']],
    on='home_gid9',
    suffixes=('_typical','_nearest')
)

car_gap['gap_car'] = (
    car_gap['avg_co2_kg_typical']
    - car_gap['avg_co2_kg_nearest']
)

In [ ]:
bike_gap['pct_change_bike'] = (
    bike_gap['gap_bike'] / bike_gap['avg_co2_kg_typical']
) * 100

pt_gap['pct_change_pt'] = (
    pt_gap['gap_pt'] / pt_gap['avg_co2_kg_typical']
) * 100

car_gap['pct_change_car'] = (
    car_gap['gap_car'] / car_gap['avg_co2_kg_typical']
) * 100

In [ ]:
import mapclassify
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
import matplotlib.patches as mpatches
import numpy as np

# -----------------------------
# Natural breaks for percentage gap
# -----------------------------
n_classes = 5
classifier = mapclassify.NaturalBreaks(car_gap['pct_change_car'], k=n_classes)
car_gap['gap_class'] = classifier.yb  # classes 0..4

# Assign colors per class
colors = plt.cm.RdYlGn_r(np.linspace(0, 1, n_classes))
car_gap['color'] = car_gap['gap_class'].apply(lambda x: colors[x])

# -----------------------------
# Plot
# -----------------------------
fig, ax = plt.subplots(figsize=(10,10))

car_gap.plot(
    color=car_gap['color'],
    linewidth=0.2,
    edgecolor='black',
    alpha=0.9,
    ax=ax
)

ctx.add_basemap(ax, source=basemaps.POSITRON)
ax.set_axis_off()
ax.set_title("CO₂ Emissions Gap nearest vs realized — Car (%)", fontsize=14)

# -----------------------------
# Legend with % ranges
# -----------------------------
labels = []
bins = classifier.bins
for i in range(len(bins)):
    if i == 0:
        low = car_gap['pct_change_car'].min()
    else:
        low = bins[i-1]
    high = bins[i]
    labels.append(f"{low:.1f}% – {high:.1f}%")

handles = [mpatches.Patch(color=colors[i], label=labels[i]) for i in range(n_classes)]
ax.legend(handles=handles, title="% Gap per Hexagon", loc="upper right", frameon=True)

plt.show()

In [ ]:
# -----------------------------
# Natural breaks for Bike
# -----------------------------
n_classes = 5
classifier = mapclassify.NaturalBreaks(bike_gap['pct_change_bike'], k=n_classes)
bike_gap['gap_class'] = classifier.yb  # classes 0..4

# Assign colors
colors = plt.cm.RdYlGn_r(np.linspace(0, 1, n_classes))
bike_gap['color'] = bike_gap['gap_class'].apply(lambda x: colors[x])

# Plot
fig, ax = plt.subplots(figsize=(10,10))
bike_gap.plot(
    color=bike_gap['color'],
    linewidth=0.2,
    edgecolor='black',
    alpha=0.9,
    ax=ax
)
ctx.add_basemap(ax, source=basemaps.POSITRON)
ax.set_axis_off()
ax.set_title("CO₂ Emissions Gap nearest vs realized — Bike (%)", fontsize=14)

# Legend with % ranges
labels = []
bins = classifier.bins
for i in range(len(bins)):
    if i == 0:
        low = bike_gap['pct_change_bike'].min()
    else:
        low = bins[i-1]
    high = bins[i]
    labels.append(f"{low:.1f}% – {high:.1f}%")

handles = [mpatches.Patch(color=colors[i], label=labels[i]) for i in range(n_classes)]
ax.legend(handles=handles, title="% Gap per Hexagon", loc="upper right", frameon=True)

plt.show()

In [ ]:
# -----------------------------
# Natural breaks for Public Transport
# -----------------------------
n_classes = 5
classifier = mapclassify.NaturalBreaks(pt_gap['pct_change_pt'], k=n_classes)
pt_gap['gap_class'] = classifier.yb  # classes 0..4

# Assign colors
colors = plt.cm.RdYlGn_r(np.linspace(0, 1, n_classes))
pt_gap['color'] = pt_gap['gap_class'].apply(lambda x: colors[x])

# Plot
fig, ax = plt.subplots(figsize=(10,10))
pt_gap.plot(
    color=pt_gap['color'],
    linewidth=0.2,
    edgecolor='black',
    alpha=0.9,
    ax=ax
)
ctx.add_basemap(ax, source=basemaps.POSITRON)
ax.set_axis_off()
ax.set_title("CO₂ Emissions Gap nearest vs realized — Public Transport (%)", fontsize=14)

# Legend with % ranges
labels = []
bins = classifier.bins
for i in range(len(bins)):
    if i == 0:
        low = pt_gap['pct_change_pt'].min()
    else:
        low = bins[i-1]
    high = bins[i]
    labels.append(f"{low:.1f}% – {high:.1f}%")

handles = [mpatches.Patch(color=colors[i], label=labels[i]) for i in range(n_classes)]
ax.legend(handles=handles, title="% Gap per Hexagon", loc="upper right", frameon=True)

plt.show()

In [ ]:
pt_gap

In [ ]:
pt_gap.sort_values("pct_change_pt")